In [1]:
#!/usr/bin/env python
from pathlib import Path
import requests, sys
from pyesgf.search import SearchConnection

SEARCH_NODE  = "https://esgf-data.dkrz.de/esg-search"
EXPERIMENTS  = ["historical", "ssp370"]#, "ssp585"]
MODEL        = "CNRM-ESM2-1"
VARIABLE     = "rsds"
TABLE        = "day"
VARIANT_PREF = ["r1i1p1f1", "r1i1p1f2"]
CHUNK_SIZE   = 1 << 20  # 1 MiB

conn = SearchConnection(SEARCH_NODE, distrib=True)

def fetch_file(url, destination):
    if destination.exists():
        return
    print(f"   ↓ Downloading {destination.name}")
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with destination.open("wb") as f:
            for chunk in response.iter_content(CHUNK_SIZE):
                f.write(chunk)

for experiment in EXPERIMENTS:
    print(f"\nExperiment: {experiment}")
    selected_ds = None
    for variant in VARIANT_PREF:
        ctx = conn.new_context(
            project="CMIP6",
            source_id=MODEL,
            experiment_id=experiment,
            variable_id=VARIABLE,
            table_id=TABLE,
            variant_label=variant,
            latest=True,
            facets="project,experiment_id,source_id,variant_label"
        )
        results = ctx.search()
        if results:
            selected_ds = results[0]
            print(f"   → Selected variant {variant}: {selected_ds.dataset_id}")
            break
    if selected_ds is None:
        print("   No matching variant found, skipping.")
        continue

    output_dir = Path("../data/cds_downloads") / experiment / MODEL
    output_dir.mkdir(parents=True, exist_ok=True)

    files = [f for f in selected_ds.file_context().search() if f.filename.endswith(".nc")]
    if not files:
        print("   No .nc files available via anonymous access.")
        continue

    for f in files:
        url  = f.download_url
        destination = output_dir / f.filename
        try:
            fetch_file(url, destination)
        except Exception as e:
            print(f"   ⚠ Error downloading {f.filename}: {e}")
            sys.exit(1)

    print(f"   ✅ Completed. Files saved in: {output_dir}")


Experiment: historical
   → Selected variant r1i1p1f2: CMIP6.CMIP.CNRM-CERFACS.CNRM-ESM2-1.historical.r1i1p1f2.day.rsds.gr.v20181206|esgf.ceda.ac.uk



-------------------------------------------------------------------------------
Warning - defaulting to search with facets=*

This behavior is kept for backward-compatibility, but ESGF indexes might not
successfully perform a distributed search when this option is used, so some
results may be missing.  For full results, it is recommended to pass a list of
facets of interest when instantiating a context object.  For example,

      ctx = conn.new_context(facets='project,experiment_id')

Only the facets that you specify will be present in the facets_counts dictionary.

This warning is displayed when a distributed search is performed while using the
facets=* default, a maximum of once per context object.  To suppress this warning,
set the environment variable ESGF_PYCLIENT_NO_FACETS_STAR_WARNING to any value
or explicitly use  conn.new_context(facets='*')

-------------------------------------------------------------------------------


   ↓ Downloading rsds_day_CNRM-ESM2-1_historical_r1i1p1f2_gr_19500101-20141231.nc


KeyboardInterrupt: 